# RSNA v4.1b smoke: 报告抽取器标签 + 无 numpy 降级 (GPU, torch cu118 规避 P100)
v4.1a 教训(2026-09-06):numpy==1.26.4 降级在 Kaggle 镜像炸了(pip resolver 报 ~15 个
numpy>=2 包冲突 + numpy 1.26.4 自身 import 崩 dtype size changed)。
v4.1b 方案:保留 numpy 2.0.2 + torch 2.2.0+cu118(实证:训练/数据加载全程 OK),
唯一崩点是 torch→numpy 的 .cpu().numpy() —— 已把 rsna_smoke.py/rsna_v3.py 全部
改成 .cpu().tolist()/纯 torch 统计,绕开 interop。
其他变更同 v4.1a:
1. 标签源弃用 barun2104(98.7% 为 0.5 死值,证伪)→ 社区规则抽取器
   rsna_report_labels.py(9 语言/按句否定极性/严重度分级);
2. smoke 本体结构不变,load_labels_smoke 调 R.load_labels 自动走新标签。
验证目标:loss 下穿 0.693 / pred_std>0.05(真正开始学习)。

In [ ]:
# P100(CUDA12 torch 无 kernel)规避:卸 torch 2.10 系,装 cu118 配对版;numpy 保持 2.0.2 不动
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "torch==2.2.0+cu118","torchvision==0.17.0+cu118",
    "--index-url","https://download.pytorch.org/whl/cu118"])
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda,
      "| numpy", numpy.__version__, "| sklearn", sklearn.__version__, "| pandas", pandas.__version__)
assert torch.cuda.is_available(), 'cuda unavailable'
# interop 自检:仅验证 numpy 侧正常(不经 torch.tensor 转换,后者实证可用)
print('numpy self-check ok:', numpy.arange(6).mean())

In [ ]:
import base64, os
parts = []
parts.append('IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiUlNOQSB2MyDnlJ/lkb3kvZPlvoEgc21va2UgdGVzdCB2MiAoMSBmb2xkIMOXIDEgZXBvY2gsIEdQVSDpop3luqbmgaLlpI3lkI7ov5DooYwpLgoKdjIg5qCh5YeGKDIwMjYtMDktMDQsIFJvb3QgQ2F1c2UgQXVkaXQg57uT6K665qCh5YeGKToKICAtIOS9k+W+gTEg5LiN5YaN55So5Y2V5LiA56Gs6ZiI5YC8LOaUueS4uuWIhue6p+aKpeWRiiArIOWIhue6p+WIpOaWrTsKICAtIOS9k+W+gTMvNCDmlLnkuLror4Hmja7pk77mjIfmoIco5LiN5YGa57ud5a+5IHBhc3MvZmFpbCzmiqXlkYrotovlir8gKyDlj4LogIPnur8pOwogIC0gcGFzcy9mYWlsIOaUueS4uuS6lOeOr+ivgeaNrumTvjogTGFiZWwg4oaSIERJQ09NIOKGkiBncmFkaWVudCDihpIgcGFyYW1ldGVyIHVwZGF0ZQogICAg4oaSIHByZWRpY3Rpb24gdmFyaWFuY2U7IOS7u+S4gOeOr+Wksei0peWNs+WBnOatouato+W8jyA1LWZvbGTjgIIKCui/kOihjChLYWdnbGUgTm90ZWJvb2sgR1BVLOaMgui9veavlOi1m+aVsOaNriArIOi9r+agh+etvuaVsOaNrumbhiArIOemu+e6v+adg+mHjSk6CiAgICAhcHl0aG9uIHJzbmFfc21va2UucHkgLS12ZXJib3NlCuS4jeWGmeato+W8jyBzdWJtaXNzaW9u44CCCiIiIgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQppbXBvcnQgaW1wb3J0bGliLnV0aWwKCkJBU0UgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKc3lzLnBhdGguaW5zZXJ0KDAsIEJBU0UpCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpzcGVjID0gaW1wb3J0bGliLnV0aWwuc3BlY19mcm9tX2ZpbGVfbG9jYXRpb24oInJzbmFfdjMiLCBvcy5wYXRoLmpvaW4oQkFTRSwgInJzbmFfdjMucHkiKSkKUiA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUoUikKCkxBQkVMUyA9IFIuTEFCRUxTClRPVEFMX0NFTExTID0gNDQwNyAqIDEyICAjIOWFqOmHjyBncmlkO+WunumZheS7pSBsZW4oZGYpKjEyIOS4uuWHhgoKIyDlj4LogIPnur8o6Z2e56Gs5oCnIHBhc3MvZmFpbCzku4XnlKjkuo7liIbnuqfliKTmlq0pClJFRiA9IHsKICAgICJoaXRfcmF0aW8iOiAwLjcsICAgICMgRElDT00g6K+75YiwIGNsaXAg55qEIHN0dWR5IOWNoOavlOWPguiAg+e6vwogICAgImxvc3NfY2FwIjogMC42NSwgICAgIyDoi6UgZmluYWwgbG9zcyDpq5jkuo7mraQs5by654OI5o+Q56S66K6t57uD5YGc5rueKGxuMj0wLjY5MykKICAgICJwcmVkX3N0ZCI6IDAuMDUsICAgICMg6aKE5rWLIHN0ZCDlj4LogIPnur8o6L+c56a7IDAgPSDmnInovpPlh7rmlrnlt64pCiAgICAiZ25vcm1fbWluIjogMC4wMSwgICAjIGdyYWRfbm9ybSDpnZ7pm7blj4LogIPnur8KICAgICJwZGVsdGFfbWluIjogMWUtNCwgICMg5Y+C5pWw57ud5a+5IGRlbHRhIOmdnumbtuWPguiAg+e6vwp9CgoKZGVmIGxvYWRfbGFiZWxzX3Ntb2tlKHJvb3QpOgogICAgIiIi5L2T5b6BMTrliIbnuqfmiqXlkYogbGFiZWwgc3VwZXJ2aXNpb24g6KaG55uWKOS4jeiuvuWUr+S4gCBwYXNzL2ZhaWwp44CCIiIiCiAgICB0cmFpbiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IikpCiAgICBvZmZpY2lhbCA9IHRyYWluW0xBQkVMU10ubm90bmEoKQogICAgZ29sZF9jZWxscyA9IGludChvZmZpY2lhbC5zdW0oKS5zdW0oKSkKICAgIGdvbGRfc3R1ZGllcyA9IGludChvZmZpY2lhbC5hbnkoYXhpcz0xKS5zdW0oKSkKCiAgICBkZiA9IFIubG9hZF9sYWJlbHMocm9vdCwgcHNldWRvX2NvbmY9MC4wLCB1c2VfcmVwb3J0X2xhYmVscz1UcnVlKQogICAgbWVyZ2VkID0gZGZbTEFCRUxTXS5ub3RuYSgpCiAgICBtZXJnZWRfY2VsbHMgPSBpbnQobWVyZ2VkLnN1bSgpLnN1bSgpKQogICAgIyDlr7npvZDliKTmlq06cHNldWRvIOWhq+WFheagvCA9IG1lcmdlZCDkuK3lrpjmlrnljp/mnKwgTmFOIOeahOS9jee9rihsb2FkX2xhYmVscyDpobrluo/kuI4gdHJhaW4g5LiA6Ie0KQogICAgcHNldWRvX2ZpbGxlZCA9IChtZXJnZWQudmFsdWVzICYgfm9mZmljaWFsLnZhbHVlcykKICAgIHBzZXVkb19jZWxscyA9IGludChwc2V1ZG9fZmlsbGVkLnN1bSgpKQogICAgbl9zdHVkaWVzX3BzZXVkbyA9IGludChwc2V1ZG9fZmlsbGVkLmFueShheGlzPTEpLnN1bSgpKQogICAgY292ZXJhZ2UgPSBtZXJnZWRfY2VsbHMgLyBtYXgoMSwgbGVuKGRmKSAqIGxlbihMQUJFTFMpKQogICAgcGVyX3N0dWR5ID0gbWVyZ2VkX2NlbGxzIC8gbWF4KDEsIGxlbihkZikpCgogICAgcHJpbnQoZiJb5L2T5b6BMV0gdG90YWwgdGFyZ2V0IGNlbGxzID0ge2xlbihkZikgKiBsZW4oTEFCRUxTKX0gKHtsZW4oZGYpfSBzdHVkaWVzIMOXIHtsZW4oTEFCRUxTKX0pIikKICAgIHByaW50KGYiW+S9k+W+gTFdIGdvbGQgdmFsaWQgY2VsbHMgICA9IHtnb2xkX2NlbGxzfSAoe2dvbGRfc3R1ZGllc30gc3R1ZGllcywg5a6Y5pa5KSIpCiAgICBwcmludChmIlvkvZPlvoExXSBwc2V1ZG8gdmFsaWQgY2VsbHMgPSB7cHNldWRvX2NlbGxzfSAoe25fc3R1ZGllc19wc2V1ZG99IHN0dWRpZXMg6KKrIHBzZXVkbyDloavlhYUpIikKICAgIHByaW50KGYiW+S9k+W+gTFdIG1lcmdlZCB2YWxpZCBjZWxscyA9IHttZXJnZWRfY2VsbHN9IikKICAgIHByaW50KGYiW+S9k+W+gTFdIHZhbGlkIGNvdmVyYWdlICAgICA9IHtjb3ZlcmFnZTouNGZ9ICh7bWVyZ2VkX2NlbGxzfS97bGVuKGRmKSAqIGxlbihMQUJFTFMpfSkiKQogICAgcHJpbnQoZiJb5L2T5b6BMV0g5q+PIHN0dWR5IOW5s+WdhyB2YWxpZCA9IHtwZXJfc3R1ZHk6LjJmfSDkuKogdGFyZ2V0IikKCiAgICAjIOWIhue6p+WIpOaWrSjpnZ7llK/kuIAgcGFzcy9mYWlsKQogICAgaWYgbWVyZ2VkX2NlbGxzIDw9IGdvbGRfY2VsbHMgKyA1MDoKICAgICAgICBsZXZlbCA9ICLinYwg5Z+65pys5Y+v56Gu6K6kIHBzZXVkbyDmnKrov5vlhaXorq3nu4MobWVyZ2VkIOKJiCBnb2xkKSIKICAgIGVsaWYgY292ZXJhZ2UgPiAwLjk6CiAgICAgICAgbGV2ZWwgPSAi4pyFIOWHoOS5juWFqOimhueblijiiaU5MCUgZ3JpZCkiCiAgICBlbGlmIG5fc3R1ZGllc19wc2V1ZG8gPiAxMDAwOgogICAgICAgIGxldmVsID0gIuKchSBwc2V1ZG8g5bey6L+b5YWl6K6t57uDKOimhuebluaVsOWNgyBzdHVkeSkiCiAgICBlbHNlOgogICAgICAgIGxldmVsID0gIuKaoO+4jyBwc2V1ZG8g6YOo5YiG6L+b5YWlKOS7heaVsOWNg+S7peS4iyBzdHVkeSks6ZyA5qOA5p+lIG1lcmdlIOaYr+WQpuWksemFjSIKICAgIHByaW50KGYiW+S9k+W+gTFdIOWIhue6p+WIpOaWrToge2xldmVsfSIpCiAgICByZXR1cm4geyJnb2xkX2NlbGxzIjogZ29sZF9jZWxscywgInBzZXVkb19jZWxscyI6IHBzZXVkb19jZWxscywKICAgICAgICAgICAgIm1lcmdlZF9jZWxscyI6IG1lcmdlZF9jZWxscywgImNvdmVyYWdlIjogY292ZXJhZ2UsCiAgICAgICAgICAgICJuX3N0dWRpZXNfcHNldWRvIjogbl9zdHVkaWVzX3BzZXVkbywgImxldmVsIjogbGV2ZWwsCiAgICAgICAgICAgICJkZiI6IGRmfQoKCmRlZiBkYXRhX3Ntb2tlKHJvb3QsIHNlcmllc19kZiwgcGxhbmUpOgogICAgIiIi5L2T5b6BMjpESUNPTSDmmK/lkKbnnJ/nmoTor7vliLAo6YeH5qC35YmNIDMyIHN0dWR5LOWQq+aAu+aVsC/lkb3kuK0v56m6L+avlOS+iynjgIIiIiIKICAgIGZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IEdyb3VwS0ZvbGQKICAgIHRyYWluX2NzdiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IikpCiAgICBna2YgPSBHcm91cEtGb2xkKG5fc3BsaXRzPTUpCiAgICB0ciwgdmEgPSBuZXh0KGl0ZXIoZ2tmLnNwbGl0KHRyYWluX2NzdiwgZ3JvdXBzPXRyYWluX2NzdlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnZhbHVlcykpKQogICAgdHJfc3R1ZGllcyA9IHRyYWluX2Nzdi5pbG9jW3RyXVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpWzozMl0KICAgIHZhX3N0dWRpZXMgPSB0cmFpbl9jc3YuaWxvY1t2YV1bIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKVs6MTZdCiAgICBkY20gPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluX3NlcmllcyIpCiAgICB0cl9kcyA9IFIuS25lZURhdGFzZXQodHJfc3R1ZGllcywgTm9uZSwgZGNtLCBzZXJpZXNfZGYsIHBsYW5lLAogICAgICAgICAgICAgICAgICAgICAgICAgIGhhc190YXJnZXRzPUZhbHNlLCBuX2NsaXBzPTIpCiAgICB2YV9kcyA9IFIuS25lZURhdGFzZXQodmFfc3R1ZGllcywgTm9uZSwgZGNtLCBzZXJpZXNfZGYsIHBsYW5lLAogICAgICAgICAgICAgICAgICAgICAgICAgIGhhc190YXJnZXRzPUZhbHNlLCBuX2NsaXBzPTIpCiAgICAjIOWFqOmDqOmHh+agtyBzdHVkeSDpg73op6blj5HliqDovb0oMzIrMTY9NDgs5bCP5qC35pysKQogICAgZm9yIHVpZCBpbiB0cl9zdHVkaWVzOgogICAgICAgIHRyX2RzLl9sb2FkX3N0dWR5KHVpZCkKICAgIGZvciB1aWQgaW4gdmFfc3R1ZGllczoKICAgICAgICB2YV9kcy5fbG9hZF9zdHVkeSh1aWQpCiAgICB0aCwgZWgsIGF0ID0gdHJfZHMuc3RhdHMoKQogICAgdmgsIGV2LCBfID0gdmFfZHMuc3RhdHMoKQogICAgdF9zYW1wbGVkID0gdGggKyBlaAogICAgdl9zYW1wbGVkID0gdmggKyBldgogICAgdF9oaXRfcmF0aW8gPSB0aCAvIG1heCgxLCB0X3NhbXBsZWQpCiAgICB2X2hpdF9yYXRpbyA9IHZoIC8gbWF4KDEsIHZfc2FtcGxlZCkKICAgIHByaW50KGYiW+S9k+W+gTJdIHRyYWluIOmHh+agtyB7dF9zYW1wbGVkfSBzdHVkeTogRElDT00g5ZG95LitIHt0aH0gLyDnqboge2VofSAvICIKICAgICAgICAgIGYi5ZG95Lit546HIHt0X2hpdF9yYXRpbzouMmZ9ICjlnYcge2F0Oi4xZn0gY2xpcC9zdHVkeSkiKQogICAgcHJpbnQoZiJb5L2T5b6BMl0gdmFsICAg6YeH5qC3IHt2X3NhbXBsZWR9IHN0dWR5OiBESUNPTSDlkb3kuK0ge3ZofSAvIOepuiB7ZXZ9IC8gIgogICAgICAgICAgZiLlkb3kuK3njocge3ZfaGl0X3JhdGlvOi4yZn0iKQogICAgcHJpbnQoZiJb5L2T5b6BMl0g5Y+C6ICD57q/5ZG95Lit546HIHtSRUZbJ2hpdF9yYXRpbyddfTvoi6Xlkb3kuK3njofiiYgwIOKGkiDovpPlhaXlhajpm7Yv6Lev5b6E6ZSZLOiuree7g+W/heWBnCIpCiAgICByZXR1cm4gdF9oaXRfcmF0aW8KCgpkZWYgdHJhaW5fc21va2Uocm9vdCwgc2VyaWVzX2RmLCBwbGFuZSwgbGFiZWxzX2RmLCBlcG9jaHM9Myk6CiAgICAiIiLkvZPlvoEzLzQvNToxIGZvbGQgw5cgTiBlcG9jaHMo6buY6K6kIDMpLOivgeaNrumTvuaMh+aghyjkuI3lgZrljZXkuIDnu53lr7kgcGFzcy9mYWlsKeOAggoKICAgIHY0OmVwb2NocyDku44gMSDihpIgMyjllK/kuIDlj5jph48pLOWFtuS9meWFqOmDqOWGu+e7k+OAguavjyBlcG9jaCDmiZPljbAKICAgIGxvc3MvdmFsaWQtc2tpcHBlZC9vcHRpbWl6ZXIgc3RlcHMvZ3JhZF9ub3JtL0xS44CC5rOo5oSP5pysIHNtb2tlIOWIu+aEj+S4jeW4pgogICAgc2NoZWR1bGVyKGxyIOaBkiAzZS00KSzkuI4gdjMg5q2j5byP6K6t57uDKOavjyBlcG9jaCDmnKvlsL4gQ29zaW5lQW5uZWFsaW5nTFIKICAgIHN0ZXAg5LiA5qyhKeeahOW3ruW8guW3suWcqOmdmeaAgeajgOafpeS4reehruiupCzpnZ4gYnVn44CCCiAgICAiIiIKICAgIGZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IEdyb3VwS0ZvbGQKICAgIHRyYWluX2NzdiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IikpCiAgICBna2YgPSBHcm91cEtGb2xkKG5fc3BsaXRzPTUpCiAgICB0ciwgdmEgPSBuZXh0KGl0ZXIoZ2tmLnNwbGl0KHRyYWluX2NzdiwgZ3JvdXBzPXRyYWluX2NzdlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnZhbHVlcykpKQogICAgdHJfc3R1ZGllcyA9IHRyYWluX2Nzdi5pbG9jW3RyXVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpCiAgICB2YV9zdHVkaWVzID0gdHJhaW5fY3N2Lmlsb2NbdmFdWyJTdHVkeUluc3RhbmNlVUlEIl0udG9saXN0KCkKICAgIGRjbSA9IG9zLnBhdGguam9pbihyb290LCAidHJhaW5fc2VyaWVzIikKICAgIHRyX2RzID0gUi5LbmVlRGF0YXNldCh0cl9zdHVkaWVzLCBsYWJlbHNfZGYsIGRjbSwgc2VyaWVzX2RmLCBwbGFuZSwgbl9jbGlwcz0yKQogICAgdmFfZHMgPSBSLktuZWVEYXRhc2V0KHZhX3N0dWRpZXMsIGxhYmVsc19kZiwgZGNtLCBzZXJpZXNfZGYsIHBsYW5lLCBuX2NsaXBzPTIpCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBtb2RlbCA9IFIuS25lZU1vZGVsKHByZXRyYWluZWQ9RmFsc2UsIG9mZmxpbmVfcGF0aHM9Ui5fc2Nhbl9vZmZsaW5lX3dlaWdodHMoKSwKICAgICAgICAgICAgICAgICAgICAgICAgY2hhbm5lbHM9MiAqIFIuQ0xJUCkudG8oZGV2aWNlKQogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0zZS00LCB3ZWlnaHRfZGVjYXk9MWUtNCkKICAgIGNyaXQgPSBubi5CQ0VXaXRoTG9naXRzTG9zcyhyZWR1Y3Rpb249Im5vbmUiKQogICAgbHJfbm93ID0gM2UtNCAgIyBzbW9rZSDml6Agc2NoZWR1bGVyLGxyIOaBkuWumijkuI4gdjMgc21va2Ug5LiA6Ie0KQoKICAgIGxvYWRlciA9IHRvcmNoLnV0aWxzLmRhdGEuRGF0YUxvYWRlcih0cl9kcywgYmF0Y2hfc2l6ZT04LCBzaHVmZmxlPVRydWUsIG51bV93b3JrZXJzPTApCiAgICBtb2RlbC50cmFpbigpCgogICAgIyDorq3nu4PliY3lj4LmlbDlv6vnhaco5YiG5LiJ57uEOmNvbnYxIC8gbGF5ZXI0IC8gaGVhZCzliIbliKvmiqXlkYrlkIToh6ogZGVsdGEpCiAgICBzbmFwMCA9IHsiY29udjEiOiBbXSwgImxheWVyNCI6IFtdLCAiaGVhZCI6IFtdfQogICAgZm9yIG4sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIGlmIG5vdCBwLnJlcXVpcmVzX2dyYWQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbi5zdGFydHN3aXRoKCJiYWNrYm9uZS5jb252MSIpOgogICAgICAgICAgICBzbmFwMFsiY29udjEiXS5hcHBlbmQoKG4sIHAuZGV0YWNoKCkuY2xvbmUoKSkpCiAgICAgICAgZWxpZiAibGF5ZXI0IiBpbiBuOgogICAgICAgICAgICBzbmFwMFsibGF5ZXI0Il0uYXBwZW5kKChuLCBwLmRldGFjaCgpLmNsb25lKCkpKQogICAgICAgIGVsaWYgbi5zdGFydHN3aXRoKCJoZWFkIik6CiAgICAgICAgICAgIHNuYXAwWyJoZWFkIl0uYXBwZW5kKChuLCBwLmRldGFjaCgpLmNsb25lKCkpKQogICAgbG9zc2VzID0gW10gICAgICAgICAgIyDlhajpg6ggc3RlcCDnmoQgbG9zcyjot6ggZXBvY2gs6aaWL+acq+eUqCkKICAgIGdub3JtID0gW10gICAgICAgICAgICMg5YWo6YOoIHN0ZXAg55qEIGdyYWQgbm9ybQogICAgZXBfc3RhdHMgPSBbXSAgICAgICAgIyDmr48gZXBvY2gg55qEIHtsb3NzX21lYW4sIHZhbGlkLCBza2lwcGVkLCBzdGVwcywgZ3JhZF9tZWFuLCBscn0KICAgIHQwID0gdGltZS50aW1lKCkKICAgIGZvciBlcCBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIGVwX2xvc3NlcyA9IFtdCiAgICAgICAgZXBfZ25vcm0gPSBbXQogICAgICAgIHRvdGFsX2JhdGNoZXMgPSAwCiAgICAgICAgdmFsaWRfYmF0Y2hlcyA9IDAKICAgICAgICBmb3IgeCwgeSwgdmFsaWQgaW4gbG9hZGVyOgogICAgICAgICAgICB0b3RhbF9iYXRjaGVzICs9IDEKICAgICAgICAgICAgeCwgeSwgdmFsaWQgPSB4LnRvKGRldmljZSksIHkudG8oZGV2aWNlKSwgdmFsaWQudG8oZGV2aWNlKQogICAgICAgICAgICBpZiB2YWxpZC5zdW0oKSA9PSAwOgogICAgICAgICAgICAgICAgY29udGludWUgICMgc2tpcHBlZCBiYXRjaDog5peg5Lu75L2V5pyJ5pWI5qCH562+CiAgICAgICAgICAgIHZhbGlkX2JhdGNoZXMgKz0gMQogICAgICAgICAgICBvcHQuemVyb19ncmFkKCkKICAgICAgICAgICAgb3V0ID0gbW9kZWwoeCkKICAgICAgICAgICAgbG9zcyA9IChjcml0KG91dCwgeSkgKiB2YWxpZCkuc3VtKCkgLyB2YWxpZC5zdW0oKQogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgZ24gPSBzdW0ocC5ncmFkLm5vcm0oKS5pdGVtKCkgKiogMiBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkKICAgICAgICAgICAgICAgICAgICAgaWYgcC5ncmFkIGlzIG5vdCBOb25lKSAqKiAwLjUKICAgICAgICAgICAgZXBfZ25vcm0uYXBwZW5kKGduKQogICAgICAgICAgICBlcF9sb3NzZXMuYXBwZW5kKGxvc3MuaXRlbSgpKQogICAgICAgICAgICBvcHQuc3RlcCgpCiAgICAgICAgbG9zc2VzLmV4dGVuZChlcF9sb3NzZXMpCiAgICAgICAgZ25vcm0uZXh0ZW5kKGVwX2dub3JtKQogICAgICAgIHNraXBwZWRfYmF0Y2hlcyA9IHRvdGFsX2JhdGNoZXMgLSB2YWxpZF9iYXRjaGVzCiAgICAgICAgZXBfc3RhdHMuYXBwZW5kKHsKICAgICAgICAgICAgImVwb2NoIjogZXAgKyAxLCAibG9zc19tZWFuIjogZmxvYXQobnAubWVhbihlcF9sb3NzZXMpKSBpZiBlcF9sb3NzZXMgZWxzZSBmbG9hdCgibmFuIiksCiAgICAgICAgICAgICJ0b3RhbCI6IHRvdGFsX2JhdGNoZXMsICJ2YWxpZCI6IHZhbGlkX2JhdGNoZXMsICJza2lwcGVkIjogc2tpcHBlZF9iYXRjaGVzLAogICAgICAgICAgICAiZ3JhZF9tZWFuIjogZmxvYXQobnAubWVhbihlcF9nbm9ybSkpIGlmIGVwX2dub3JtIGVsc2UgMC4wLAogICAgICAgICAgICAibHIiOiBscl9ub3d9KQogICAgICAgIHByaW50KGYiW+S9k+W+gTNdW2Vwb2NoIHtlcCArIDF9L3tlcG9jaHN9XSBsb3NzX21lYW4ge2VwX3N0YXRzWy0xXVsnbG9zc19tZWFuJ106LjRmfSB8ICIKICAgICAgICAgICAgICBmInZhbGlkIHt2YWxpZF9iYXRjaGVzfS97dG90YWxfYmF0Y2hlc30gc2tpcHBlZCB7c2tpcHBlZF9iYXRjaGVzfSBzdGVwcyB7dmFsaWRfYmF0Y2hlc30gfCAiCiAgICAgICAgICAgICAgZiJncmFkX25vcm1fbWVhbiB7ZXBfc3RhdHNbLTFdWydncmFkX21lYW4nXTouNGZ9IHwgbHIge2xyX25vdzouMmV9ICIKICAgICAgICAgICAgICBmIih7dGltZS50aW1lKCkgLSB0MDouMGZ9cykiLCBmbHVzaD1UcnVlKQogICAgb3B0aW1pemVyX3N0ZXBzID0gc3VtKHNbInZhbGlkIl0gZm9yIHMgaW4gZXBfc3RhdHMpCiAgICB0b3RhbF9iYXRjaGVzID0gc3VtKHNbInRvdGFsIl0gZm9yIHMgaW4gZXBfc3RhdHMpCiAgICBza2lwcGVkX2JhdGNoZXMgPSBzdW0oc1sic2tpcHBlZCJdIGZvciBzIGluIGVwX3N0YXRzKQogICAgdmFsaWRfYmF0Y2hlcyA9IHN1bShzWyJ2YWxpZCJdIGZvciBzIGluIGVwX3N0YXRzKQogICAgaW5pdGlhbF9sb3NzID0gbG9zc2VzWzBdIGlmIGxvc3NlcyBlbHNlIGZsb2F0KCJuYW4iKQogICAgZmluYWxfbG9zcyA9IGxvc3Nlc1stMV0gaWYgbG9zc2VzIGVsc2UgZmxvYXQoIm5hbiIpCgogICAgIyDliIbnjq/lj4LmlbAgZGVsdGEoY29udjEgLyBsYXllcjQgLyBoZWFkLOiuree7g+WJjeWQjuWQhOiHque7neWvueW3rikKICAgIHBkZWx0YSA9IHt9CiAgICBmb3IgZ3JwIGluICgiY29udjEiLCAibGF5ZXI0IiwgImhlYWQiKToKICAgICAgICBkID0gMC4wCiAgICAgICAgZm9yIG4sIHAwIGluIHNuYXAwW2dycF06CiAgICAgICAgICAgIHAxID0gZGljdChtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkpW25dCiAgICAgICAgICAgIGQgKz0gZmxvYXQoKHAxLmRldGFjaCgpIC0gcDApLmFicygpLnN1bSgpKQogICAgICAgIHBkZWx0YVtncnBdID0gZAogICAgdG90YWxfZGVsdGEgPSBzdW0ocGRlbHRhLnZhbHVlcygpKQoKICAgIHByaW50KGYiW+S9k+W+gTVdIOWIhueOr+WPguaVsOe7neWvuSBkZWx0YTogY29udjE9e3BkZWx0YVsnY29udjEnXTouNmZ9ICAiCiAgICAgICAgICBmImxheWVyND17cGRlbHRhWydsYXllcjQnXTouNmZ9ICBoZWFkPXtwZGVsdGFbJ2hlYWQnXTouNmZ9ICAiCiAgICAgICAgICBmIijlkIjorqEge3RvdGFsX2RlbHRhOi42Zn07IOWPguiAgzog5ZCE546vPntSRUZbJ3BkZWx0YV9taW4nXX0gPSDor6XlsYLnnJ/lnKjmm7TmlrApIikKCiAgICAjIOS9k+W+gTQ66aqM6K+B6ZuGIGxvZ2l0cy9wcmVkIOe7n+iuoSjpmZDliY0gNjQgc3R1ZHks6YG/5YWN5YWo6YePIHZhbCBESUNPTSDliqDovb3mi5bplb8pCiAgICBtb2RlbC5ldmFsKCkKICAgIGxvZ2l0c19hbGwsIHByZWRzID0gW10sIFtdCiAgICB2YWxfc3ViID0gdmFfc3R1ZGllc1s6NjRdCiAgICB2YV9zdWJfZHMgPSBSLktuZWVEYXRhc2V0KHZhbF9zdWIsIE5vbmUsIGRjbSwgc2VyaWVzX2RmLCBwbGFuZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaGFzX3RhcmdldHM9RmFsc2UsIG5fY2xpcHM9MikKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGZvciB4LCBfLCBfIGluIHRvcmNoLnV0aWxzLmRhdGEuRGF0YUxvYWRlcih2YV9zdWJfZHMsIGJhdGNoX3NpemU9OCwgc2h1ZmZsZT1GYWxzZSk6CiAgICAgICAgICAgIHggPSB4LnRvKGRldmljZSkKICAgICAgICAgICAgbGcgPSBtb2RlbCh4KQogICAgICAgICAgICAjIOazqOaEjzrkuI3nlKggLmNwdSgpLm51bXB5KCkg4oCU4oCUIHRvcmNoIDIuMi4wK2N1MTE4IOe8luS6jiBudW1weTEueCwKICAgICAgICAgICAgIyBLYWdnbGUgbnVtcHkyLjAuMiDkuIvor6XmlrnlkJEgaW50ZXJvcCDltKkoIk51bXB5IGlzIG5vdCBhdmFpbGFibGUiKSwKICAgICAgICAgICAgIyDnu5/orqHlhajotbAgdG9yY2gsdG9saXN0L2Zsb2F0KCkg5LiN57uPIG51bXB544CCCiAgICAgICAgICAgIGxvZ2l0c19hbGwuYXBwZW5kKGxnKQogICAgICAgICAgICBwcmVkcy5hcHBlbmQodG9yY2guc2lnbW9pZChsZykpCiAgICBsb2dpdHMgPSB0b3JjaC5jYXQobG9naXRzX2FsbCwgZGltPTApIGlmIGxvZ2l0c19hbGwgZWxzZSB0b3JjaC56ZXJvcygoMSwgMSksIGRldmljZT1kZXZpY2UpCiAgICBwcmVkID0gdG9yY2guY2F0KHByZWRzLCBkaW09MCkgaWYgcHJlZHMgZWxzZSB0b3JjaC56ZXJvcygoMSwgMSksIGRldmljZT1kZXZpY2UpCgogICAgcHJpbnQoZiJb5L2T5b6BM10gdG90YWxfYmF0Y2hlcz17dG90YWxfYmF0Y2hlc30gIHZhbGlkX2JhdGNoZXM9e3ZhbGlkX2JhdGNoZXN9ICAiCiAgICAgICAgICBmInNraXBwZWRfYmF0Y2hlcz17c2tpcHBlZF9iYXRjaGVzfSAgb3B0aW1pemVyX3N0ZXBzPXtvcHRpbWl6ZXJfc3RlcHN9IikKICAgIHByaW50KGYiW+S9k+W+gTNdIGluaXRpYWxfbG9zcz17aW5pdGlhbF9sb3NzOi40Zn0gIGZpbmFsX2xvc3M9e2ZpbmFsX2xvc3M6LjRmfSAgIgogICAgICAgICAgZiIo55CG5oOzOiBmaW5hbCA8IGluaXRpYWw7IOWPguiAgzog6Iul57uI5YC85LuN4omIMC42OTMg4oaSIOiuree7g+WBnOa7nikiKQogICAgaWYgZ25vcm06CiAgICAgICAgcHJpbnQoZiJb5L2T5b6BM10gZ3JhZF9ub3JtIG1lYW49e25wLm1lYW4oZ25vcm0pOi40Zn0gbWluPXtucC5taW4oZ25vcm0pOi40Zn0gIgogICAgICAgICAgICAgIGYibWF4PXtucC5tYXgoZ25vcm0pOi40Zn0iKQogICAgZWxzZToKICAgICAgICBwcmludCgiW+S9k+W+gTNdIGdyYWRfbm9ybTog5peg5pyJ5pWIIHN0ZXAiKQogICAgcHJpbnQoZiJb5L2T5b6BNF0gKHZhbCDliY0ge2xlbih2YWxfc3ViKX0gc3R1ZHkpIGxvZ2l0cyBtZWFuPXtsb2dpdHMubWVhbigpOi40Zn0gIgogICAgICAgICAgZiJzdGQ9e2xvZ2l0cy5zdGQoKTouNGZ9IikKICAgIHByaW50KGYiW+S9k+W+gTRdIHByZWQgbWVhbj17cHJlZC5tZWFuKCk6LjRmfSBzdGQ9e3ByZWQuc3RkKCk6LjRmfSAiCiAgICAgICAgICBmIijlj4LogIM6IHN0ZD57UkVGWydwcmVkX3N0ZCddfSA9IOi+k+WHuuacieaWueW3rikiKQoKICAgIHJldHVybiB7InRvdGFsX2JhdGNoZXMiOiB0b3RhbF9iYXRjaGVzLCAidmFsaWRfYmF0Y2hlcyI6IHZhbGlkX2JhdGNoZXMsCiAgICAgICAgICAgICJza2lwcGVkX2JhdGNoZXMiOiBza2lwcGVkX2JhdGNoZXMsICJvcHRpbWl6ZXJfc3RlcHMiOiBvcHRpbWl6ZXJfc3RlcHMsCiAgICAgICAgICAgICJpbml0aWFsX2xvc3MiOiBpbml0aWFsX2xvc3MsICJmaW5hbF9sb3NzIjogZmluYWxfbG9zcywKICAgICAgICAgICAgImdub3JtIjogZ25vcm0sICJsb2dpdHNfc3RkIjogZmxvYXQobG9naXRzLnN0ZCgpKSwKICAgICAgICAgICAgInByZWRfc3RkIjogZmxvYXQocHJlZC5zdGQoKSksICJwZGVsdGEiOiB0b3RhbF9kZWx0YSwKICAgICAgICAgICAgInBkZWx0YV9ieV9ncm91cCI6IHBkZWx0YSwgImVwX3N0YXRzIjogZXBfc3RhdHN9CgoKZGVmIG1haW4oKToKICAgIHJvb3QsIF8gPSBSLnJlc29sdmVfcGF0aHMoKQogICAgdHJhaW5fY3N2ID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKHJvb3QsICJ0cmFpbi5jc3YiKSkKICAgIHNlcmllc19kZiA9IHBkLnJlYWRfY3N2KG9zLnBhdGguam9pbihyb290LCAidHJhaW5fc2VyaWVzLmNzdiIpKQogICAgcGxhbmUgPSAiU2FnaXR0YWwiCiAgICBwcmludChmIltzbW9rZV0gcm9vdD17cm9vdH0gIHBsYW5lPXtwbGFuZX0iKQogICAgdDAgPSB0aW1lLnRpbWUoKQoKICAgICMgPT09PT09PT09PT09IOS6lOeOr+ivgeaNrumTviA9PT09PT09PT09PT0KICAgIGV2aWRlbmNlID0ge30KCiAgICAjIOeOrzE6IExhYmVsIHN1cGVydmlzaW9uCiAgICBsYWIgPSBsb2FkX2xhYmVsc19zbW9rZShyb290KQogICAgZXZpZGVuY2VbImxhYmVsIl0gPSAobGFiWyJtZXJnZWRfY2VsbHMiXSA+IGxhYlsiZ29sZF9jZWxscyJdICsgNTApCiAgICBwcmludChmIlvor4Hmja7pk74xIExhYmVsXSB7J+KchScgaWYgZXZpZGVuY2VbJ2xhYmVsJ10gZWxzZSAn4p2MJ30gIgogICAgICAgICAgZiIobWVyZ2VkIHtsYWJbJ21lcmdlZF9jZWxscyddfSA+IGdvbGQge2xhYlsnZ29sZF9jZWxscyddfSkiKQoKICAgICMg546vMjogRElDT00gaW5wdXQKICAgIGhpdF9yYXRpbyA9IGRhdGFfc21va2Uocm9vdCwgc2VyaWVzX2RmLCBwbGFuZSkKICAgIGV2aWRlbmNlWyJkaWNvbSJdID0gaGl0X3JhdGlvID4gMC4wCiAgICBwcmludChmIlvor4Hmja7pk74yIERJQ09NXSB7J+KchScgaWYgZXZpZGVuY2VbJ2RpY29tJ10gZWxzZSAn4p2MJ30g5ZG95Lit546HPXtoaXRfcmF0aW86LjJmfSIpCgogICAgIyDnjq8zLTU6IOiuree7g+WKqOaAgSh2NDogZXBvY2hzPTMs5ZSv5LiA5Y+Y6YePO3YzIOS4uiAxLOWPr+eUqCAtLWVwb2NocyDlpI3njrApCiAgICBpbXBvcnQgYXJncGFyc2UKICAgIF9hcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIF9hcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MykKICAgIF9hcC5hZGRfYXJndW1lbnQoIi0tdmVyYm9zZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcmdzX3Ntb2tlLCBfID0gX2FwLnBhcnNlX2tub3duX2FyZ3MoKQogICAgdHIgPSB0cmFpbl9zbW9rZShyb290LCBzZXJpZXNfZGYsIHBsYW5lLAogICAgICAgICAgICAgICAgICAgICBOb25lIGlmIG5vdCBldmlkZW5jZVsibGFiZWwiXSBlbHNlIGxhYi5nZXQoImRmIiwgTm9uZSksCiAgICAgICAgICAgICAgICAgICAgIGVwb2Nocz1hcmdzX3Ntb2tlLmVwb2NocykKICAgIGV2aWRlbmNlWyJncmFkaWVudCJdID0gYm9vbCh0clsiZ25vcm0iXSkgYW5kIG1heCh0clsiZ25vcm0iXSkgPiBSRUZbImdub3JtX21pbiJdCiAgICBldmlkZW5jZVsicGFyYW1fdXBkYXRlIl0gPSB0clsicGRlbHRhIl0gPiBSRUZbInBkZWx0YV9taW4iXQogICAgZXZpZGVuY2VbInByZWRfdmFyIl0gPSB0clsicHJlZF9zdGQiXSA+IFJFRlsicHJlZF9zdGQiXQogICAgcHJpbnQoZiJb6K+B5o2u6ZO+MyBHcmFkaWVudF0geyfinIUnIGlmIGV2aWRlbmNlWydncmFkaWVudCddIGVsc2UgJ+KdjCd9ICIKICAgICAgICAgIGYiZ3JhZF9ub3JtX21heD17bWF4KHRyWydnbm9ybSddKTouNGZ9IiBpZiB0clsiZ25vcm0iXSBlbHNlICJb6K+B5o2u6ZO+MyBHcmFkaWVudF0g4p2MIOaXoCBzdGVwIikKICAgIHByaW50KGYiW+ivgeaNrumTvjQgUGFyYW1VcGRhdGVdIHsn4pyFJyBpZiBldmlkZW5jZVsncGFyYW1fdXBkYXRlJ10gZWxzZSAn4p2MJ30gIgogICAgICAgICAgZiJkZWx0YT17dHJbJ3BkZWx0YSddOi42Zn0iKQogICAgcHJpbnQoZiJb6K+B5o2u6ZO+NSBQcmVkVmFyXSB7J+KchScgaWYgZXZpZGVuY2VbJ3ByZWRfdmFyJ10gZWxzZSAn4p2MJ30gIgogICAgICAgICAgZiJwcmVkX3N0ZD17dHJbJ3ByZWRfc3RkJ106LjRmfSIpCgogICAgcHJpbnQoZiJcbj09PT09IOS6lOeOr+ivgeaNrumTvuaAu+e7kyAo6ICX5pe2IHt0aW1lLnRpbWUoKSAtIHQwOi4wZn1zKSA9PT09PSIpCiAgICBjaGFpbiA9IFsibGFiZWwiLCAiZGljb20iLCAiZ3JhZGllbnQiLCAicGFyYW1fdXBkYXRlIiwgInByZWRfdmFyIl0KICAgIGZvciBrIGluIGNoYWluOgogICAgICAgIHByaW50KGYiICB7azoxM3N9OiB7J+KchScgaWYgZXZpZGVuY2UuZ2V0KGspIGVsc2UgJ+KdjCd9IikKICAgIGFsbF9vayA9IGFsbChldmlkZW5jZS5nZXQoaykgZm9yIGsgaW4gY2hhaW4pCiAgICBmYWlsZWQgPSBbayBmb3IgayBpbiBjaGFpbiBpZiBub3QgZXZpZGVuY2UuZ2V0KGspXQogICAgaWYgYWxsX29rOgogICAgICAgIHByaW50KCJcbuKchSDkupTnjq/lhajov4cg4oaSIOWPr+S7pei/m+WFpeato+W8jyA1LWZvbGTDl04tZXBvY2gg5YWo6YeP6K6t57uDIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIlxu4p2MIOS6lOeOr+acquWFqOi/hyzlpLHotKXnjq86ICIgKyAiLCAiLmpvaW4oZmFpbGVkKSkKICAgICAgICBwcmludCgi5oyJ5a6h6K6h5oql5ZGK5a+55bqUIFJvb3QgQ2F1c2Ug5L+u5aSN5a+55bqU546v6IqCLOWLv+ebtOaOpei3keWFqOmHjzsiCiAgICAgICAgICAgICAgIuiLpeS7heS4quWIq+eOr+W8sSzlhYjop4Llr5/otovlir/lho3lhrPlrpoiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK')
open("/kaggle/working/rsna_smoke.py","wb").write(base64.b64decode("".join(parts)))
print("rsna_smoke.py ->", os.path.getsize("/kaggle/working/rsna_smoke.py")//1024, "KB")

In [ ]:
import base64, os
parts = []
parts.append('...')
open("/kaggle/working/rsna_v3.py","wb").write(base64.b64decode("".join(parts)))
print("rsna_v3.py ->", os.path.getsize("/kaggle/working/rsna_v3.py")//1024, "KB")

In [ ]:
import base64, os
parts = []
parts.append('...')
open("/kaggle/working/rsna_report_labels.py","wb").write(base64.b64decode("".join(parts)))
print("rsna_report_labels.py ->", os.path.getsize("/kaggle/working/rsna_report_labels.py")//1024, "KB")

In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("=== 运行 v4.1b smoke (1 fold x 3 epochs, GPU, 报告抽取器标签) ===")
r = subprocess.run([sys.executable,"-u","rsna_smoke.py","--epochs","3"], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("STDERR:", r.stderr[-3000:])